In [1]:
import json
import os
from PIL import Image
from datasets import Dataset
from sklearn.metrics import accuracy_score,f1_score, recall_score
from transformers import AutoImageProcessor
from torchvision.transforms import RandomResizedCrop, Compose, Normalize, ToTensor
from transformers import AutoModelForImageClassification, TrainingArguments, Trainer
from transformers import DefaultDataCollator
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
path = 'D:/code/geovit/dataset'
 
 
def gen(path):
    image_json = os.path.join(path, "image.json")
    with open(image_json, 'r') as f:
        # 读取JSON数据
        data = json.load(f)
    for key, value in data.items():
        imagePath = os.path.join(path, "image")
        imagePath = os.path.join(imagePath, key)
        image = Image.open(imagePath)
        yield {'image': image, 'label': value}
 
 
def get_label(path):
    label_json = os.path.join(path, "label.json")
    with open(label_json, 'r') as f:
        # 读取JSON数据
        data = json.load(f)
    label2id, id2label = dict(), dict()
    for key, value in data.items():
        label2id[key] = str(value)
        id2label[str(value)] = key
    return label2id, id2label
 
 
ds = Dataset.from_generator(gen, gen_kwargs={"path": path})
ds = ds.train_test_split(test_size=0.2)
label2id, id2label = get_label(path)
 
checkpoint = 'google/vit-base-patch16-224'
image_processor = AutoImageProcessor.from_pretrained(checkpoint)
 
normalize = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)
size = (
    image_processor.size["shortest_edge"]
    if "shortest_edge" in image_processor.size
    else (image_processor.size["height"], image_processor.size["width"])
)
_transforms = Compose([RandomResizedCrop(size), ToTensor(), normalize])
 
 
def transforms(examples):
    examples["pixel_values"] = [_transforms(img.convert("RGB")) for img in examples["image"]]
    del examples["image"]
    return examples
 
 
ds = ds.with_transform(transforms)
 
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels,preds,average="weighted")
    acc = accuracy_score(labels,preds)
    recall = recall_score(labels,preds,average="weighted")
    return {"accuracy":acc,"f1":f1, "recall": recall}
 
 
model = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=2619,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True)
 
training_args = TrainingArguments(
    output_dir="models",
    remove_unused_columns=False,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=30,
    warmup_ratio=0.1,
    logging_steps=10,
    greater_is_better=True,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)
 
data_collator = DefaultDataCollator()
 
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    tokenizer=image_processor,
    compute_metrics=compute_metrics,
)
 
trainer.train()

Found cached dataset generator (C:/Users/liuxiao/.cache/huggingface/datasets/generator/default-78b6e564a16744ea/0.0.0)
'HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /google/vit-base-patch16-224/resolve/main/preprocessor_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000001FE18CA25F0>, 'Connection to huggingface.co timed out. (connect timeout=10)'))' thrown while requesting HEAD https://huggingface.co/google/vit-base-patch16-224/resolve/main/preprocessor_config.json
'HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /google/vit-base-patch16-224/resolve/main/config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000001FE18CA3970>, 'Connection to huggingface.co timed out. (connect timeout=10)'))' thrown while requesting HEAD https://huggingface.co/google/vit-base-patch16-224/resolve/main/config.json
'HTTPSConnectionPool(host='huggingfac

Epoch,Training Loss,Validation Loss,Accuracy,F1,Recall
1,5.675400,5.604881,0.132300,0.057536,0.132300
2,4.531800,4.520218,0.241700,0.158640,0.241700
3,3.911600,3.916080,0.310700,0.238429,0.310700
4,3.358200,3.598700,0.359200,0.295279,0.359200
5,2.685100,3.403561,0.372000,0.313954,0.372000
6,2.828600,3.287946,0.386500,0.336725,0.386500
7,2.317600,3.179059,0.408100,0.359398,0.408100
8,2.012600,3.101388,0.420200,0.378722,0.420200
9,1.896200,3.100588,0.417800,0.377061,0.417800
10,1.753600,3.091913,0.425000,0.387611,0.425000


D:\software\anaconda\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\software\anaconda\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\software\anaconda\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\software\anaconda\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall is ill-defined and 

TrainOutput(global_step=18750, training_loss=1.5023989681053163, metrics={'train_runtime': 68977.5356, 'train_samples_per_second': 17.397, 'train_steps_per_second': 0.272, 'total_flos': 9.51715089948672e+19, 'train_loss': 1.5023989681053163, 'epoch': 30.0})